# Session 22: Unsloth 기반 파인튜닝

## 🎯 Unsloth 소개 (2x 빠른 학습, 60% 메모리 절약)

**Unsloth**는 LLM 파인튜닝을 **2배 빠르게**, **60% 적은 메모리**로 수행할 수 있게 해주는 최적화 라이브러리입니다.

### Unsloth의 핵심 특징

| 특징 | 설명 |
|------|------|
| 🚀 2배 빠른 학습 | 커스텀 CUDA 커널로 학습 속도 향상 |
| 💾 60% 메모리 절약 | 최적화된 메모리 관리 |
| 🔧 간편한 사용 | 기존 HuggingFace 코드와 호환 |
| 📦 GGUF 변환 지원 | Ollama 바로 연동 가능 |
| 🎯 다양한 모델 지원 | Qwen, LLaMA, Mistral, Gemma 등 |

### Unsloth vs 일반 Transformers

| 항목 | 일반 Transformers | Unsloth |
|------|------------------|----------|
| 학습 속도 | 기준 (1x) | ~2x 빠름 |
| VRAM 사용 | 기준 | ~60% 절약 |
| RTX 4060 최대 모델 | 3B QLoRA | 7B QLoRA 안정 |
| GGUF 변환 | 별도 도구 필요 | 내장 지원 |

### 학습 목표

- ✅ Unsloth 설치 및 기본 사용법
- ✅ FastLanguageModel로 모델 로드 및 LoRA 설정
- ✅ SFTTrainer로 학습 실행
- ✅ 학습 전후 비교
- ✅ GGUF 변환 및 Ollama 연동

## 1️⃣ Unsloth 설치 및 설정

⚠️ Unsloth는 별도 설치가 필요합니다. 이미 설치되어 있지 않다면 아래 셀을 실행하세요.

In [ ]:
# Unsloth 설치 (필요시 주석 해제)
# !pip install unsloth
# 또는 특정 버전
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

print("⚠️ Unsloth가 설치되어 있지 않다면 위의 주석을 해제하고 실행하세요.")
print("📌 설치 후 런타임을 재시작해야 할 수 있습니다.")

In [ ]:
# 라이브러리 임포트
import torch
import gc
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print_gpu_memory("시작")

In [ ]:
# Unsloth 임포트 확인
try:
    from unsloth import FastLanguageModel
    print("✅ Unsloth 임포트 성공!")
    UNSLOTH_AVAILABLE = True
except ImportError:
    print("⚠️ Unsloth가 설치되어 있지 않습니다.")
    print("📌 일반 Transformers로 대체하여 진행합니다.")
    print("📌 Unsloth 설치: pip install unsloth")
    UNSLOTH_AVAILABLE = False
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, TaskType

## 2️⃣ Unsloth로 모델 로드 (FastLanguageModel)

Unsloth의 `FastLanguageModel`은 모델 로드와 4bit 양자화를 한 번에 처리합니다.

In [ ]:
# 모델 설정
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # Unsloth는 메모리 효율적이므로 3B 가능
MAX_SEQ_LENGTH = 1024
OUTPUT_DIR = "./output/unsloth_finetuning"

if UNSLOTH_AVAILABLE:
    print("⏳ Unsloth FastLanguageModel로 모델 로딩 중...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None,  # 자동 감지
    )
    print(f"✅ Unsloth 모델 로드 완료!")
else:
    print("⏳ 일반 Transformers로 모델 로딩 중...")
    # Unsloth 없을 때 fallback으로 1.5B 사용
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config,
        device_map="auto", trust_remote_code=True
    )
    model.gradient_checkpointing_enable()
    print(f"✅ 일반 Transformers 모델 로드 완료 ({MODEL_NAME})")

print(f"📌 모델: {MODEL_NAME}")
print_gpu_memory("모델 로드 후")

## 3️⃣ LoRA 어댑터 설정

Unsloth의 `get_peft_model`은 자동으로 최적의 target_modules를 선택합니다.

In [ ]:
if UNSLOTH_AVAILABLE:
    print("⏳ Unsloth LoRA 어댑터 적용 중...")
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        lora_dropout=0,  # Unsloth는 dropout=0 권장 (최적화)
        bias="none",
        use_gradient_checkpointing="unsloth",  # Unsloth 최적화 체크포인팅
        random_state=42,
    )
    print("✅ Unsloth LoRA 적용 완료!")
else:
    print("⏳ 일반 PEFT LoRA 어댑터 적용 중...")
    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_config)
    print("✅ 일반 PEFT LoRA 적용 완료!")

# 학습 가능 파라미터 확인
model.print_trainable_parameters()
print_gpu_memory("LoRA 적용 후")

## 4️⃣ 데이터 준비 및 포맷팅

In [ ]:
from datasets import Dataset

# Alpaca 데이터 로드
data_path = "../data/samples/alpaca_ko_sample.json"
with open(data_path, "r", encoding="utf-8") as f:
    alpaca_data = json.load(f)

print(f"✅ 데이터 로드: {len(alpaca_data)}개 샘플")

# Chat Template 변환
def format_to_chat(sample):
    instruction = sample["instruction"]
    input_text = sample.get("input", "")
    output_text = sample["output"]
    
    user_content = f"{instruction}\n\n{input_text}" if input_text else instruction
    messages = [
        {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다. 사용자의 질문에 정확하고 친절하게 답변해주세요."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output_text}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return text

formatted_texts = [format_to_chat(s) for s in alpaca_data]
dataset = Dataset.from_dict({"text": formatted_texts})

print(f"✅ 데이터 포맷팅 완료: {len(dataset)}개 샘플")
print(f"\n--- 포맷팅 예시 ---")
print(dataset[0]["text"][:400])

## 5️⃣ SFTTrainer로 학습 실행

In [ ]:
# 학습 전 응답 저장
TEST_PROMPTS = [
    "LoRA 파인튜닝의 장점을 설명하세요.",
    "다음 문장을 영어로 번역하세요.\n\n오늘 날씨가 정말 좋습니다.",
    "Python에서 리스트와 튜플의 차이점을 설명하세요.",
    "GPU 메모리 부족 시 해결 방법을 알려주세요.",
    "트랜스포머의 Attention 메커니즘을 쉽게 설명하세요."
]

print("📋 학습 전 응답 수집 중...")

if UNSLOTH_AVAILABLE:
    FastLanguageModel.for_inference(model)
else:
    model.eval()

before_responses = []
for prompt in TEST_PROMPTS:
    messages = [
        {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=150,
            do_sample=False, temperature=1.0
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    before_responses.append(response)
    print(f"\n🔹 {prompt[:50]}...")
    print(f"   {response[:120]}")

print("\n✅ 학습 전 응답 수집 완료")

In [ ]:
from trl import SFTTrainer, SFTConfig

# SFTTrainer 설정
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,        # RTX 4060 안전 설정
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    save_strategy="epoch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to="none",
    optim="adamw_8bit",  # 8bit Adam으로 메모리 추가 절약
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("✅ SFTTrainer 초기화 완료")
print(f"📌 Optimizer: adamw_8bit")
print(f"📌 fp16: {sft_config.fp16}, bf16: {sft_config.bf16}")
print_gpu_memory("학습 시작 전")

In [ ]:
# 학습 실행
print("🚀 학습 시작! (Unsloth 최적화)" if UNSLOTH_AVAILABLE else "🚀 학습 시작! (일반 Transformers)")
print("="*60)

start_time = time.time()
train_result = trainer.train()
training_time = time.time() - start_time

print("="*60)
print(f"✅ 학습 완료!")
print(f"📌 소요 시간: {training_time:.1f}초 ({training_time/60:.1f}분)")
print(f"📌 Final Loss: {train_result.training_loss:.4f}")
print(f"📌 Total Steps: {train_result.global_step}")
print_gpu_memory("학습 완료")

## 6️⃣ 학습 전후 비교

In [ ]:
# 학습 후 응답 생성
print("="*60)
print("📊 학습 전후 비교 (Before vs After)")
print("="*60)

if UNSLOTH_AVAILABLE:
    FastLanguageModel.for_inference(model)
else:
    model.eval()

for i, prompt in enumerate(TEST_PROMPTS):
    messages = [
        {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=150,
            do_sample=False, temperature=1.0
        )
    after_response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    print(f"\n{'='*60}")
    print(f"🔹 질문: {prompt[:70]}")
    print(f"\n  [Before] {before_responses[i][:200]}")
    print(f"\n  [After]  {after_response[:200]}")

print(f"\n{'='*60}")

## 7️⃣ Unsloth vs 일반 Transformers 속도 비교

Unsloth의 최적화 효과를 확인합니다.

In [ ]:
# 속도 비교 (이론적 수치 + 실측)
print("="*60)
print("📊 Unsloth vs 일반 Transformers 비교")
print("="*60)

if UNSLOTH_AVAILABLE:
    print(f"\n🚀 실측 결과 (Unsloth):")
    print(f"   학습 시간: {training_time:.1f}초")
    print(f"   Final Loss: {train_result.training_loss:.4f}")
    print(f"\n📌 일반 Transformers 대비 예상 개선:")
    print(f"   속도: ~2배 빠름")
    print(f"   메모리: ~60% 절약")
else:
    print(f"\n📌 일반 Transformers 실측 결과:")
    print(f"   학습 시간: {training_time:.1f}초")
    print(f"   Final Loss: {train_result.training_loss:.4f}")
    print(f"\n📌 Unsloth 사용 시 예상 개선:")
    print(f"   예상 학습 시간: ~{training_time/2:.1f}초 (2배 빠름)")
    print(f"   예상 메모리 절약: ~60%")

print(f"""
\n📋 Unsloth 최적화 원리:
   1️⃣ 커스텀 CUDA 커널 - 어텐션/행렬곱 최적화
   2️⃣ 메모리 효율적 역전파 - 불필요한 텐서 즉시 해제
   3️⃣ RoPE 임베딩 최적화 - 위치 인코딩 캐싱
   4️⃣ Cross Entropy Loss 최적화 - 인플레이스 연산
   5️⃣ 자동 Mixed Precision - 최적 dtype 자동 선택
""")

## 8️⃣ GGUF 변환 및 Ollama 연동

학습된 모델을 GGUF 포맷으로 변환하면 Ollama에서 바로 사용할 수 있습니다.

In [ ]:
# LoRA 어댑터 저장
save_path = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ LoRA 어댑터 저장: {save_path}")

# 저장 크기 확인
total_size = sum(
    os.path.getsize(os.path.join(root, f))
    for root, dirs, files in os.walk(save_path)
    for f in files
)
print(f"📌 어댑터 크기: {total_size/1024/1024:.1f} MB")

In [ ]:
# GGUF 변환 (Unsloth 사용 시)
if UNSLOTH_AVAILABLE:
    print("⏳ GGUF 변환 중... (Q4_K_M 양자화)")
    try:
        gguf_path = os.path.join(OUTPUT_DIR, "gguf")
        model.save_pretrained_gguf(
            gguf_path,
            tokenizer,
            quantization_method="q4_k_m"  # 가장 많이 사용되는 양자화
        )
        
        # GGUF 파일 크기 확인
        gguf_size = sum(
            os.path.getsize(os.path.join(root, f))
            for root, dirs, files in os.walk(gguf_path)
            for f in files
        )
        print(f"✅ GGUF 변환 완료!")
        print(f"📌 GGUF 파일 크기: {gguf_size/1024/1024:.0f} MB")
        print(f"📌 저장 위치: {gguf_path}")
    except Exception as e:
        print(f"⚠️ GGUF 변환 실패: {e}")
        print("📌 llama.cpp가 필요할 수 있습니다.")
else:
    print("ℹ️ Unsloth 없이 GGUF 변환하려면 llama.cpp를 사용해야 합니다.")
    print("")
    print("📋 llama.cpp로 GGUF 변환 방법:")
    print("   1. git clone https://github.com/ggerganov/llama.cpp")
    print("   2. cd llama.cpp && make")
    print("   3. python convert_hf_to_gguf.py <모델 경로> --outtype q4_k_m")

In [ ]:
# Ollama 연동 가이드
print("="*60)
print("📋 Ollama 연동 가이드")
print("="*60)

modelfile_content = f"""# Modelfile 예시
FROM {OUTPUT_DIR}/gguf/unsloth.Q4_K_M.gguf

TEMPLATE \"\"\"<|im_start|>system
{{{{.System}}}}<|im_end|>
<|im_start|>user
{{{{.Prompt}}}}<|im_end|>
<|im_start|>assistant
\"\"\"

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER stop "<|im_end|>"

SYSTEM \"당신은 유용한 AI 어시스턴트입니다.\"
"""

print(modelfile_content)
print("\n📋 Ollama 실행 명령어:")
print("   1. 위 내용을 Modelfile로 저장")
print("   2. ollama create my-finetuned-model -f Modelfile")
print("   3. ollama run my-finetuned-model")

In [ ]:
# GPU 메모리 정리
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print_gpu_memory("메모리 정리 후")
print("✅ GPU 메모리 정리 완료")

## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

| 항목 | 내용 |
|------|------|
| Unsloth | 2배 빠른 학습, 60% 메모리 절약 |
| FastLanguageModel | 모델 로드 + 양자화 + LoRA를 한 번에 |
| GGUF 변환 | `save_pretrained_gguf`로 간편 변환 |
| Ollama 연동 | Modelfile 작성으로 바로 서빙 |

### 핵심 포인트

- 🎯 **Unsloth는 기존 코드와 호환**되면서 성능을 크게 향상시킵니다
- 🎯 **RTX 4060에서 7B 모델**도 안정적으로 학습 가능
- 🎯 **GGUF 변환 → Ollama** 파이프라인으로 학습부터 배포까지 한 번에
- 🎯 **dropout=0**이 Unsloth 권장 설정 (최적화 호환)
- 🎯 **adamw_8bit** 옵티마이저로 추가 메모리 절약 가능

### Unsloth를 쓰면 좋은 경우

- ✅ GPU 메모리가 제한적 (8~16GB)
- ✅ 빠른 실험 반복이 필요한 경우
- ✅ 학습 후 GGUF/Ollama 배포 계획이 있는 경우
- ✅ QLoRA로 7B+ 모델을 학습하고 싶은 경우

### 다음 단계

- ➡️ **Notebook 22**: Tool Calling 개념 - API 기반 도구 호출 이해
- ➡️ **Notebook 23-24**: Tool Calling 데이터 생성 및 파인튜닝